In [ ]:
"""
FILE 6: SORTING AND FUNCTIONAL PATTERNS
===========================================
Sorting bahut algorithms ka pehla step hota hai (binary search, two pointer,
merge intervals, greedy). Yeh file sorting ke saath saath Python ke chote
"functional" tricks bhi dikhati hai jo code ko chota aur clean bana dete hain.
"""

from functools import cmp_to_key
from collections import Counter


# -------------------------------------------------------------
# 1) BASIC SORTING
# -------------------------------------------------------------
nums = [3, 1, 4, 1, 5]
nums.sort()                    # IN-PLACE sort -> original list badal jaati hai
sorted_nums = sorted(nums)      # NAYI sorted list banti hai, original waisi hi rehti hai

nums.sort(reverse=True)          # descending order mein sort
sorted(nums, reverse=True)         # descending, naya list


# -------------------------------------------------------------
# 2) key= SE CUSTOM SORTING
# -------------------------------------------------------------
# key ek function leta hai jo har element se "compare karne wali value" nikaalta hai

intervals = [[1, 3], [2, 6], [8, 10]]
intervals.sort(key=lambda x: x[0])   # start time ke hisaab se sort (merge intervals)

words = ["banana", "kiwi", "apple"]
words.sort(key=lambda w: len(w))       # length ke hisaab se sort

# MULTI-KEY SORT: pehle length, phir (agar length same ho) alphabet se sort
words.sort(key=lambda w: (len(w), w))
# Python tuple ko LEXICOGRAPHICALLY compare karta hai isliye yeh trick chalta hai

nums.sort(key=abs)   # absolute value ke hisaab se sort

# frequency ke hisaab se sort (sabse zyada frequent pehle):
freq = Counter(nums)
nums.sort(key=lambda x: -freq[x])   # negative isliye taaki descending order aa jaaye

# EK KEY ascending, DUSRI descending karni ho:
students = [("amit", 90), ("riya", 85), ("sam", 90)]
students.sort(key=lambda s: (-s[1], s[0]))   # score DESC (negate), naam ASC


# -------------------------------------------------------------
# 3) cmp_to_key - jab key= kaafi nahi hota
# -------------------------------------------------------------
# "Largest Number" problem: numbers ko is tarah jodo ki sabse bada number bane
def compare(a, b):
    if a + b > b + a:
        return -1   # a pehle aana chahiye
    elif a + b < b + a:
        return 1     # b pehle aana chahiye
    return 0

nums_int = [3, 30, 34, 5, 9]
nums_str = [str(n) for n in nums_int]
nums_str.sort(key=cmp_to_key(compare))
result = "".join(nums_str)


# -------------------------------------------------------------
# 4) SORT STABILITY - Python ka sort STABLE hai (TimSort algorithm)
# -------------------------------------------------------------
# Matlab: agar 2 elements compare mein EQUAL hain, to unka ORIGINAL order
# waisa hi rehta hai. Isse pata chalta hai:

# Method 1 (BEST): ek hi sort call, tuple key ke saath
# students.sort(key=lambda s: (s.grade, s.name))

# Method 2 (yeh bhi chalega, kyunki sort stable hai): 2 alag sorts
# students.sort(key=lambda s: s.name)     # PEHLE secondary key se sort karo
# students.sort(key=lambda s: s.grade)     # PHIR primary key se sort karo (stable rehta hai)


# ===============================================================
# FUNCTIONAL PATTERNS - chote Python tricks jo code short bana dete hain
# ===============================================================

# -------------------------------------------------------------
# 5) LIST COMPREHENSION vs GENERATOR EXPRESSION
# -------------------------------------------------------------
squares = [x**2 for x in range(10)]     # [ ] -> POORI list turant memory mein banti hai

total = sum(x**2 for x in range(10))      # ( ) -> GENERATOR, lazy hai, list nahi banti
# Generator memory bachata hai jab tumhe sirf ek baar iterate karna ho.

has_negative = any(x < 0 for x in [1, -2, 3])   # koi ek bhi True ho to True
all_positive = all(x > 0 for x in [1, 2, 3])      # SAB True hone chahiye

# Pehla match dhundna:
first_even = next((x for x in [1, 3, 4, 5] if x % 2 == 0), None)   # None agar na mile


# -------------------------------------------------------------
# 6) zip() - MULTIPLE LISTS EK SAATH GHUMANA (parallel iteration)
# -------------------------------------------------------------
list1 = [1, 2, 3]
list2 = ['a', 'b', 'c']
for a, b in zip(list1, list2):
    pass   # (1,'a'), (2,'b'), (3,'c')

# Adjacent elements compare karna (jaise "sorted hai ya nahi" check):
nums_check = [1, 2, 5, 4]
for curr, nxt in zip(nums_check, nums_check[1:]):
    if curr > nxt:
        pass   # not sorted!

# Matrix TRANSPOSE karna (rows <-> columns):
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
transposed = list(zip(*matrix))   # [(1,4,7), (2,5,8), (3,6,9)]

# UNZIP karna (paired data ko alag karna):
pairs = [(1, 'a'), (2, 'b'), (3, 'c')]
nums_z, letters = zip(*pairs)   # nums_z=(1,2,3), letters=('a','b','c')


# -------------------------------------------------------------
# 7) reversed() - bina naya list banaye ulta ghumana
# -------------------------------------------------------------
for val in reversed([1, 2, 3]):
    pass   # naya list NAHI banti, sirf iterator hota hai - memory efficient

reversed_str = "".join(reversed("hello"))
# Ya seedha slicing: reversed_str = "hello"[::-1]   -> yeh bhi common hai


# -------------------------------------------------------------
# 8) MULTIPLE ASSIGNMENT AUR UNPACKING TRICKS
# -------------------------------------------------------------
a, b = b, a if False else (1, 2)   # (dummy, dikhane ke liye) swap ka pattern
first, *rest = [1, 2, 3, 4]          # first=1, rest=[2, 3, 4]
first2, *middle, last = [1, 2, 3, 4]   # first2=1, middle=[2, 3], last=4
_, val, _ = (1, 2, 3)                    # sirf beech wali value chahiye, baaki ignore

d_sample = {"a": 1, "b": 2}
for i, (key, value) in enumerate(d_sample.items()):
    pass


# -------------------------------------------------------------
# 9) COUNTER ARITHMETIC - SLIDING WINDOW PROBLEMS MEIN
# -------------------------------------------------------------
def findAnagrams(s, p):
    if len(p) > len(s):
        return []

    p_count = Counter(p)
    window = Counter(s[:len(p)])
    result = []

    if window == p_count:   # Counter ko SEEDHA compare kar sakte ho (== se)
        result.append(0)

    for i in range(len(p), len(s)):
        window[s[i]] += 1                  # naya character window mein jodo
        old_char = s[i - len(p)]
        window[old_char] -= 1               # purana character hatao
        if window[old_char] == 0:
            del window[old_char]              # 0-count entries hatao, taaki compare saaf rahe

        if window == p_count:
            result.append(i - len(p) + 1)

    return result


# -------------------------------------------------------------
# 10) map() aur filter() - type conversion ke liye handy
# -------------------------------------------------------------
raw_input_line = "1 2 3 4"
nums_from_input = list(map(int, raw_input_line.split()))   # strings -> ints
result_str = "".join(map(str, nums_from_input))               # ints -> string (join se pehle)

# List comprehension generally zyada readable maani jaati hai, lekin type
# conversion ke liye map() bhi kaafi common hai.